# Семинар 3. Случайные величины и распределения

**Курс «Теория вероятностей», ИТМО.** Лекция 3: дискретные и непрерывные случайные величины, функция распределения, плотность, основные распределения, применение в анализе данных.

Сегодня всё, что на лекции было формулами, мы **получим руками на симуляциях** и сравним с теорией, а в конце применим распределения к данным: логам запросов и реальному датасету.

| Время | Блок | Что делаем |
|---|---|---|
| 0–10 мин | 0 | Запускаем ноутбук, вспоминаем `numpy` и `scipy.stats` |
| 10–25 мин | 1 | Случайная величина как функция исхода: два кубика |
| 25–40 мин | 2 | Функция распределения и ECDF |
| 40–55 мин | 3 | Непрерывные величины: плотность, гистограмма, $P(X=x)=0$ |
| 55–65 мин | 4 | Галерея распределений `scipy.stats` и типичные ловушки |
| 65–75 мин | 5 | Генерируем распределения сами: метод обратной функции |
| 75–90 мин | 6 | Распределения в данных: нагрузка на сервер и цены на жильё |

Задания отмечены **✏️ Задание** и строкой `# TODO` в коде. Там, где стоит `...`, нужно дописать код.

## Как открыть ноутбук в Google Colab

1. Скачайте файл `03-seminar-random-variables-itmo.ipynb` на свой компьютер.
2. Откройте [colab.research.google.com](https://colab.research.google.com) и войдите в свой Google-аккаунт.
3. Выберите «Файл → Загрузить блокнот» (File → Upload notebook), перейдите на вкладку «Загрузить» и выберите скачанный файл. Можно просто перетащить файл в окно.
4. Ноутбук сохранится на вашем Google Диске в папке «Colab Notebooks». В следующий раз открывайте его оттуда, а не загружайте заново, иначе потеряете свои правки.
5. «Среда выполнения → Выполнить все» (Runtime → Run all) запускает все ячейки; одна ячейка запускается сочетанием `Shift+Enter`.

Ничего устанавливать не нужно: `numpy`, `scipy`, `pandas` и `matplotlib` в Colab уже есть.

## 0. Подготовка

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats

rng = np.random.default_rng(2026)   # фиксируем генератор, чтобы результаты повторялись
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
print("numpy", np.__version__, "| scipy", scipy.__version__)

**Памятка по `scipy.stats`.** Каждое распределение — объект с одинаковыми методами:

| Метод | Что возвращает |
|---|---|
| `pmf(k)` / `pdf(x)` | вероятность точки (дискретные) / плотность (непрерывные) |
| `cdf(x)` | $F(x)=P(X\le x)$ |
| `sf(x)` | $P(X>x)=1-F(x)$ |
| `ppf(q)` | квантиль: $F^{-1}(q)$ |
| `rvs(size, random_state)` | выборка |
| `mean()`, `var()`, `median()` | характеристики |

In [ ]:
X = stats.binom(n=10, p=0.3)
print("P(X = 3)   =", X.pmf(3))
print("P(X <= 3)  =", X.cdf(3))
print("P(X > 3)   =", X.sf(3))
print("медиана    =", X.median())
print("5 значений =", X.rvs(size=5, random_state=rng))

---
## 1. Случайная величина — функция исхода

Бросаем два кубика. Элементарный исход — пара $\omega=(i,j)$, всего 36 равновероятных исходов. Случайная величина $X(\omega)=i+j$ — сумма очков.

Сначала посчитаем ряд распределения **точно**, перебрав все исходы, а потом **смоделируем** эксперимент.

In [ ]:
outcomes = [(i, j) for i in range(1, 7) for j in range(1, 7)]   # Ω
X_of = lambda w: w[0] + w[1]                                     # X: Ω → R

values = np.arange(2, 13)
exact = np.array([sum(X_of(w) == x for w in outcomes) for x in values]) / 36
print(pd.DataFrame({"x": values, "P(X=x)": exact.round(4), "×36": (exact * 36).round().astype(int)}).T)
print("Сумма вероятностей:", exact.sum())

In [ ]:
N = 100_000
dice = rng.integers(1, 7, size=(N, 2))     # N бросков пары кубиков
sums = dice.sum(axis=1)                     # значения X

freq = np.bincount(sums, minlength=13)[2:] / N
plt.bar(values, freq, alpha=0.6, label=f"частоты, {N:,} бросков".replace(",", " "))
plt.plot(values, exact, "ko-", label="точный ряд")
plt.xticks(values); plt.xlabel("x"); plt.ylabel("P(X = x)"); plt.legend(); plt.show()

**Закон больших чисел в действии.** Как частота события $\{X=7\}$ приближается к вероятности $6/36$ с ростом числа бросков:

In [ ]:
running = np.cumsum(sums == 7) / np.arange(1, N + 1)
plt.semilogx(np.arange(1, N + 1), running, label="частота {X = 7}")
plt.axhline(6 / 36, color="red", ls="--", label="P(X = 7) = 1/6")
plt.xlabel("число бросков (лог. шкала)"); plt.legend(); plt.show()

### ✏️ Задание 1.1. Максимум двух кубиков

На том же $\Omega$ определим другую случайную величину: $Y(\omega)=\max(i,j)$.

1. Найдите точный ряд распределения $Y$ перебором исходов (массив `exact_Y` для значений $1,\dots,6$).
2. Найдите частоты по симуляции `dice` (массив `freq_Y`).
3. Сравните с формулой $P(Y=k)=\dfrac{2k-1}{36}$. Откуда она берётся?

In [ ]:
k = np.arange(1, 7)

# TODO: точный ряд распределения Y = max(i, j) перебором outcomes
exact_Y = ...

# TODO: частоты по симуляции dice (подсказка: dice.max(axis=1) и np.bincount)
freq_Y = ...

print(pd.DataFrame({"k": k, "точно": exact_Y, "симуляция": freq_Y, "(2k-1)/36": (2 * k - 1) / 36}).round(4))

*Вопрос для обсуждения.* $X$ и $Y$ заданы на одном и том же $\Omega$. Почему у них разные распределения? Придумайте на этом $\Omega$ случайную величину, у которой распределение **такое же**, как у $Y$, но которая не равна $Y$.

---
## 2. Функция распределения и ECDF

Функция распределения $F(x)=P(X\le x)$. Для выборки её оценка — **эмпирическая функция распределения**
$$
F_n(x) = \frac{\#\{i: x_i\le x\}}{n}.
$$

### ✏️ Задание 2.1. Своя ECDF

Напишите функцию `ecdf(sample, x)`, которая возвращает $F_n(x)$ для массива точек `x`. Подсказка: отсортируйте выборку и используйте `np.searchsorted(..., side="right")`, либо (медленнее) сравнение с бродкастингом.

In [ ]:
def ecdf(sample, x):
    # TODO: вернуть массив F_n(x) той же формы, что x
    ...

# проверка на простом примере: выборка 1, 2, 2, 3
print(ecdf(np.array([1, 2, 2, 3]), np.array([0, 1, 1.5, 2, 2.9, 3, 10])))   # ожидаем [0, .25, .25, .75, .75, 1, 1]

In [ ]:
grid = np.linspace(0, 14, 1401)
F_exact = lambda x: np.array([exact[values <= xx].sum() for xx in np.atleast_1d(x)])

plt.step(grid, F_exact(grid), where="post", lw=3, alpha=0.4, label="точная F(x)")
plt.step(grid, ecdf(sums[:200], grid), where="post", label="ECDF, n = 200")
plt.xlabel("x"); plt.ylabel("F(x) = P(X ≤ x)"); plt.legend(); plt.show()

### ✏️ Задание 2.2. Интервалы и скачки

Для суммы двух кубиков посчитайте **по точной $F$** и проверьте **на симуляции** `sums`:

1. $P(4< X\le 7)$;
2. $P(4\le X\le 7)$ — осторожно, это не $F(7)-F(4)$;
3. $P(X=7)$ как высоту скачка $F(7)-F(6{,}999)$.

In [ ]:
# TODO: три вероятности по точной F (функция F_exact) и по симуляции
p1_theory, p1_sim = ..., ...
p2_theory, p2_sim = ..., ...
p3_theory, p3_sim = ..., ...

print(f"P(4 < X <= 7):  {p1_theory:.4f}  vs  {p1_sim:.4f}")
print(f"P(4 <= X <= 7): {p2_theory:.4f}  vs  {p2_sim:.4f}")
print(f"P(X = 7):       {p3_theory:.4f}  vs  {p3_sim:.4f}")

> **Соглашение.** В `scipy` и в нашем курсе $F(x)=P(X\le x)$. В задачнике Гмурмана $F(x)=P(X<x)$. Для дискретных величин это меняет значения $F$ в точках скачков: проверьте, что `stats.binom(10, 0.3).cdf(3)` — это $P(X\le 3)$, а не $P(X<3)$.

---
## 3. Непрерывные величины: плотность

Гистограмма с `density=True` — эмпирическая оценка плотности: высота столбика = доля наблюдений в нём / ширина столбика. Площадь всей гистограммы равна 1.

In [ ]:
z = rng.normal(size=50_000)
x = np.linspace(-4, 4, 400)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, bins in zip(axes, [8, 40, 400]):
    ax.hist(z, bins=bins, density=True, alpha=0.6)
    ax.plot(x, stats.norm.pdf(x), "r", lw=2)
    ax.set_title(f"{bins} столбиков")
plt.show()

*Вопрос.* Почему при 400 столбиках гистограмма «шумит», а при 8 — слишком грубая? Чем в этом смысле удобнее ECDF?

**Вероятность точки равна нулю.** Сколько раз в 50 000 нормальных значений встретится ровно `0.5`? А сколько значений попадёт в интервал $(0{,}5-h;\ 0{,}5+h)$ — и как это связано с $f(0{,}5)\cdot 2h$?

In [ ]:
print("ровно 0.5:", np.sum(z == 0.5))
for h in [0.1, 0.01, 0.001]:
    share = np.mean(np.abs(z - 0.5) < h)
    print(f"h = {h:<6} доля в (0.5 ± h) = {share:.5f}   f(0.5)·2h = {stats.norm.pdf(0.5) * 2 * h:.5f}")

**Плотность может быть больше 1.** У $U(0;\,0{,}5)$ плотность равна 2. Проверьте: высота гистограммы около 2, а площадь — 1.

In [ ]:
u = rng.uniform(0, 0.5, size=50_000)
heights, edges, _ = plt.hist(u, bins=25, density=True, alpha=0.6)
plt.axhline(2, color="red", ls="--"); plt.title("U(0; 0,5)"); plt.show()
print("площадь гистограммы:", np.sum(heights * np.diff(edges)))

### ✏️ Задание 3.1. Своя плотность

Случайная величина имеет плотность $f(x)=3x^2$ на $[0;1]$.

1. На бумаге найдите $F(x)$, $P(X>0{,}5)$ и медиану.
2. Чтобы смоделировать такую величину, решите уравнение $F(x)=u$ относительно $x$. Если $U\sim U(0;1)$, то $X=F^{-1}(U)$ имеет нужное распределение (это метод обратной функции, подробнее в блоке 5).
3. Сгенерируйте 100 000 значений, постройте гистограмму с плотностью и сравните $P(X>0{,}5)$ и медиану с теорией.

In [ ]:
# TODO: впишите ответы с бумаги
p_gt_half_theory = ...
median_theory = ...

# TODO: 100 000 значений X через F^{-1}(U)
U = rng.random(100_000)
X3 = ...

xx = np.linspace(0, 1, 200)
plt.hist(X3, bins=50, density=True, alpha=0.6, label="симуляция")
plt.plot(xx, 3 * xx**2, "r", lw=2, label="f(x) = 3x²"); plt.legend(); plt.show()
print(f"P(X > 0.5): теория {p_gt_half_theory:.4f}, симуляция {np.mean(X3 > 0.5):.4f}")
print(f"медиана:    теория {median_theory:.4f}, симуляция {np.median(X3):.4f}")

---
## 4. Галерея распределений

Функция ниже рисует ряд/плотность и функцию распределения любого объекта `scipy.stats`. Меняйте параметры и смотрите, как меняется форма.

In [ ]:
def show(dist, title, lo=None, hi=None):
    lo = dist.ppf(0.001) if lo is None else lo
    hi = dist.ppf(0.999) if hi is None else hi
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    if hasattr(dist, "pmf"):
        k = np.arange(np.floor(lo), np.ceil(hi) + 1)
        axes[0].bar(k, dist.pmf(k), alpha=0.6); axes[0].set_title(f"{title}: P(X = k)")
        axes[1].step(k, dist.cdf(k), where="post")
    else:
        x = np.linspace(lo, hi, 400)
        axes[0].plot(x, dist.pdf(x)); axes[0].fill_between(x, dist.pdf(x), alpha=0.3); axes[0].set_title(f"{title}: f(x)")
        axes[1].plot(x, dist.cdf(x))
    axes[1].set_title("F(x)")
    plt.show()
    print(f"EX = {dist.mean():.3f},  DX = {dist.var():.3f},  медиана = {dist.median():.3f}")

show(stats.binom(n=20, p=0.3), "B(20; 0,3)")
show(stats.poisson(mu=4), "Pois(4)")
show(stats.geom(p=0.3), "Geom(0,3)", lo=1)
show(stats.expon(scale=1 / 2), "Exp(λ = 2)")
show(stats.norm(loc=10, scale=2), "N(10; 2²)")

**Интерактивно (по желанию).** В Colab работают виджеты: двигайте ползунки и следите, как биномиальное распределение $B(n,\lambda/n)$ приближается к пуассоновскому.

In [ ]:
try:
    from ipywidgets import interact

    @interact(n=(5, 500, 5), lam=(0.5, 10, 0.5))
    def binom_vs_poisson(n=10, lam=4):
        k = np.arange(0, 21)
        plt.bar(k, stats.poisson.pmf(k, lam), alpha=0.5, label=f"Pois({lam})")
        plt.plot(k, stats.binom.pmf(k, n, min(lam / n, 1)), "ko--", label=f"B({n}; {lam}/{n})")
        plt.legend(); plt.show()
except ImportError:
    print("ipywidgets не установлен — пропускаем интерактивную часть")

### ✏️ Задание 4.1. Правило трёх сигм — проверка симуляцией

Сгенерируйте 1 000 000 значений из $N(\mu=50,\ \sigma=8)$ и найдите долю значений в $\mu\pm\sigma$, $\mu\pm2\sigma$, $\mu\pm3\sigma$. Сравните с точными значениями через `stats.norm.cdf`.

Затем сделайте то же для **показательного** распределения с тем же средним 50 (у него $\sigma$ тоже 50). Работает ли «правило трёх сигм» для несимметричного распределения?

In [ ]:
mu, sigma = 50, 8
sample_norm = rng.normal(mu, sigma, size=1_000_000)
sample_exp = rng.exponential(scale=50, size=1_000_000)   # среднее 50, σ = 50

for kk in [1, 2, 3]:
    # TODO: доли попаданий в mu ± k·sigma для обеих выборок и точное значение для нормального
    share_norm = ...
    exact_norm = ...
    share_exp = ...
    print(f"±{kk}σ: нормальное {share_norm:.4f} (точно {exact_norm:.4f}),  показательное {share_exp:.4f}")

### ✏️ Задание 4.2. Ловушка параметризации

Аналитик хотел смоделировать время между запросами при интенсивности $\lambda=4$ запроса в минуту и написал `stats.expon(4)`. Что он получил на самом деле? Выведите `mean()` у его объекта и у правильного. Исправьте код.

In [ ]:
wrong = stats.expon(4)
# TODO: правильный объект для Exp(λ = 4)
right = ...
print("неправильно: среднее", wrong.mean(), "  носитель от", wrong.ppf(0))
print("правильно:   среднее", right.mean())

---
## 5. Генерируем распределения сами: метод обратной функции

**Теорема.** Если $U\sim U(0;1)$ и $F$ непрерывна и строго возрастает, то $X=F^{-1}(U)$ имеет функцию распределения $F$: $P(F^{-1}(U)\le x)=P(U\le F(x))=F(x)$.

Для $\mathrm{Exp}(\lambda)$: $F(x)=1-e^{-\lambda x}$, откуда $F^{-1}(u)=-\ln(1-u)/\lambda$.

**Дискретный вариант.** Разбиваем $[0;1]$ на отрезки длины $p_1,p_2,\dots$; номер отрезка, в который попало $U$, — номер значения. В коде: `np.searchsorted(np.cumsum(p), u)`.

### ✏️ Задание 5.1. Два генератора

1. `my_exponential(lam, size)` — выборка из $\mathrm{Exp}(\lambda)$ только через `rng.random`.
2. `my_discrete(values, probs, size)` — выборка из дискретного распределения с рядом `values`, `probs`.

Проверка: сравните гистограммы с теорией и посчитайте статистику Колмогорова $\sup_x|F_n(x)-F(x)|$ через `stats.kstest`.

In [ ]:
def my_exponential(lam, size):
    # TODO
    ...

def my_discrete(values, probs, size):
    # TODO
    ...

e = my_exponential(2.0, 100_000)
print("среднее:", e.mean(), "(теория 0.5)")
print(stats.kstest(e, stats.expon(scale=1 / 2).cdf))

vals, probs = np.array([0, 1, 2, 3]), np.array([0.729, 0.243, 0.027, 0.001])   # отказы 3 серверов из лекции
d = my_discrete(vals, probs, 100_000)
print("частоты:", np.bincount(d, minlength=4) / len(d), " теория:", probs)

*Вопрос.* Почему можно писать `-np.log(u) / lam` вместо `-np.log(1 - u) / lam`? (Подсказка: как распределено $1-U$?)

---
## 6. Распределения в анализе данных

### 6.1. Логи сервера: Пуассон и показательное

Смоделируем журнал запросов к сервису за 8 часов работы: запросы приходят независимо со средней интенсивностью $\lambda=4$ в минуту (пуассоновский поток). В журнале есть только **моменты прихода** запросов — как в настоящих логах.

In [ ]:
LAM, MINUTES = 4, 8 * 60
gaps = rng.exponential(scale=1 / LAM, size=LAM * MINUTES * 2)
timestamps = np.cumsum(gaps)
timestamps = timestamps[timestamps < MINUTES]           # моменты запросов, в минутах от начала
log = pd.DataFrame({"t_min": timestamps})
print(len(log), "запросов")
log.head()

Из журнала получаем две случайные величины:

- $N$ — число запросов в каждую минуту (дискретная, кандидат — Пуассон);
- $T$ — интервал между соседними запросами (непрерывная, кандидат — показательное).

In [ ]:
counts = np.bincount(log["t_min"].astype(int), minlength=MINUTES)      # N по минутам
intervals = np.diff(log["t_min"].to_numpy())                           # T

print(f"N: среднее {counts.mean():.3f}, дисперсия {counts.var(ddof=1):.3f}   (у Пуассона они равны)")
print(f"T: среднее {intervals.mean():.4f}, ст. откл. {intervals.std(ddof=1):.4f}   (у показательного они равны)")

lam_hat = counts.mean()       # оценка λ методом моментов
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
k = np.arange(0, counts.max() + 1)
axes[0].bar(k, np.bincount(counts) / MINUTES, alpha=0.6, label="данные")
axes[0].plot(k, stats.poisson.pmf(k, lam_hat), "ko-", label=f"Pois({lam_hat:.2f})")
axes[0].set_title("Запросов в минуту"); axes[0].legend()
t = np.linspace(0, intervals.max(), 300)
axes[1].hist(intervals, bins=50, density=True, alpha=0.6, label="данные")
axes[1].plot(t, stats.expon.pdf(t, scale=1 / lam_hat), "r", label=f"Exp({lam_hat:.2f})")
axes[1].set_title("Интервалы между запросами, мин"); axes[1].legend()
plt.show()

### ✏️ Задание 6.1. Вопросы эксплуатации

Сервер обрабатывает не более 7 запросов в минуту. По **модели** с оценённым $\hat\lambda$ и по **данным** (доля минут) найдите:

1. долю минут с перегрузкой ($N\ge 8$);
2. вероятность ждать следующий запрос дольше 30 секунд;
3. минимальную пропускную способность $c$ (запросов в минуту), при которой перегрузка $N>c$ случается не чаще чем в 1 % минут. Подсказка: это квантиль.

In [ ]:
# TODO
overload_model, overload_data = ..., ...
wait_model, wait_data = ..., ...
c_needed = ...

print(f"перегрузка:     модель {overload_model:.4f}, данные {overload_data:.4f}")
print(f"ждать > 30 с:   модель {wait_model:.4f}, данные {wait_data:.4f}")
print(f"нужная мощность: {c_needed} запросов в минуту")

**Отсутствие памяти на данных.** Если мы уже ждём запрос 15 секунд, с какой вероятностью придётся ждать ещё хотя бы 15? Сравните условную частоту с безусловной $P(T>0{,}25)$.

In [ ]:
s, tt = 0.25, 0.25
cond = np.mean(intervals[intervals > s] > s + tt)
uncond = np.mean(intervals > tt)
print(f"P(T > s+t | T > s) ≈ {cond:.4f},   P(T > t) ≈ {uncond:.4f},   e^(-λt) = {np.exp(-lam_hat * tt):.4f}")

*Вопрос.* В реальном сервисе днём запросов больше, чем ночью. Что тогда произойдёт с отношением «дисперсия / среднее» для $N$? Смоделируйте: пусть первые 4 часа $\lambda=2$, следующие 4 часа $\lambda=6$.

### 6.2. Реальные данные: цены на жильё в Калифорнии

В Colab в папке `sample_data` лежит датасет California Housing (данные переписи 1990 года по районам Калифорнии). Посмотрим на две колонки:

- `median_income` — медианный доход в районе (в десятках тысяч долларов);
- `median_house_value` — медианная стоимость дома, долларов.

In [ ]:
import os

PATHS = ["/content/sample_data/california_housing_train.csv",
         "https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv"]
df = None
for p in PATHS:
    try:
        if p.startswith("http") or os.path.exists(p):
            df = pd.read_csv(p)
            print("загружено:", p)
            break
    except Exception as exc:
        print("не получилось:", p, exc)
if df is None:   # запасной вариант без интернета: синтетические данные похожей формы
    print("Датасет недоступен, используем синтетические данные")
    df = pd.DataFrame({"median_income": rng.lognormal(np.log(3.5), 0.45, 17_000)})
    df["median_house_value"] = np.minimum(rng.lognormal(np.log(180_000), 0.55, 17_000), 500_001)

df[["median_income", "median_house_value"]].describe().round(2)

In [ ]:
inc = df["median_income"].to_numpy()

mu_n, sd_n = inc.mean(), inc.std(ddof=1)                     # нормальная модель
mu_l, sd_l = np.log(inc).mean(), np.log(inc).std(ddof=1)     # логнормальная модель: ln X ~ N(mu_l, sd_l²)
normal_model = stats.norm(mu_n, sd_n)
lognormal_model = stats.lognorm(s=sd_l, scale=np.exp(mu_l))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
x = np.linspace(0, inc.max(), 400)
axes[0].hist(inc, bins=60, density=True, alpha=0.5, label="данные")
axes[0].plot(x, normal_model.pdf(x), label="нормальное")
axes[0].plot(x, lognormal_model.pdf(x), label="логнормальное")
axes[0].set_title("Плотность"); axes[0].legend()

axes[1].plot(np.sort(inc), np.arange(1, len(inc) + 1) / len(inc), label="ECDF")
axes[1].plot(x, normal_model.cdf(x), "--", label="нормальное")
axes[1].plot(x, lognormal_model.cdf(x), "--", label="логнормальное")
axes[1].set_title("Функция распределения"); axes[1].legend()

q = (np.arange(1, 200) - 0.5) / 199
emp_q = np.quantile(inc, q)
axes[2].plot(normal_model.ppf(q), emp_q, ".", label="нормальное")
axes[2].plot(lognormal_model.ppf(q), emp_q, ".", label="логнормальное")
axes[2].plot([0, 15], [0, 15], "k", lw=1)
axes[2].set_xlabel("квантили модели"); axes[2].set_ylabel("квантили данных")
axes[2].set_title("QQ-график"); axes[2].legend()
plt.show()

### ✏️ Задание 6.2. Какая модель лучше?

1. Сравните долю районов с доходом выше 8 (то есть выше \$80 000) в данных и по каждой модели.
2. Сравните 99-й процентиль дохода в данных и по моделям.
3. Какую вероятность нормальная модель приписывает **отрицательному** доходу?

Сделайте вывод: какая модель лучше описывает правый хвост и почему это важно, например, для оценки доли богатых районов.

In [ ]:
# TODO
share_data, share_norm, share_lognorm = ..., ..., ...
q99_data, q99_norm, q99_lognorm = ..., ..., ...
p_negative = ...

print(f"доля > 8:   данные {share_data:.4f}, нормальная {share_norm:.4f}, логнормальная {share_lognorm:.4f}")
print(f"p99:        данные {q99_data:.2f}, нормальная {q99_norm:.2f}, логнормальная {q99_lognorm:.2f}")
print(f"P(доход < 0) по нормальной модели: {p_negative:.4f}")

### 6.3. Смешанное распределение в реальных данных

Посмотрите на `median_house_value`. При сборе данных значения выше \$500 000 записали как 500 001 — стоимость **обрезана** сверху. Получилась **смешанная** случайная величина: непрерывная часть плюс точка с положительной вероятностью.

In [ ]:
v = df["median_house_value"].to_numpy()
p_atom = np.mean(v >= 500_000)
print(f"доля районов со значением 500 001: {p_atom:.4f}")

xs = np.sort(v)
plt.plot(xs, np.arange(1, len(xs) + 1) / len(xs))
plt.axvline(500_001, color="red", ls="--")
plt.title("ECDF стоимости дома: скачок в конце — атом вероятности")
plt.xlabel("долларов"); plt.show()

*Вопрос.* Чему равна высота скачка ECDF в точке 500 001? Что пойдёт не так, если обучать на этой колонке регрессию с MSE и не знать об обрезке?

---
## Итоги

- Случайная величина — функция исхода; одна и та же $\Omega$ порождает разные величины с разными распределениями.
- $F(x)=P(X\le x)$ работает для любых величин; ECDF — её оценка по данным.
- Гистограмма с `density=True` оценивает плотность; вероятность точки у непрерывной величины равна нулю.
- `scipy.stats`: `pmf/pdf`, `cdf`, `sf`, `ppf`, `rvs`. Следите за параметризацией (`scale=1/λ`).
- Из $U(0;1)$ методом обратной функции получается любое распределение.
- Модель распределения выбирают по смыслу величины, а проверяют по ECDF и QQ-графику; хвосты важнее центра.

**Домашнее задание** — отдельный ноутбук `hw03-random-variables-itmo.ipynb` (условия и порядок сдачи — внутри).